## Misclassified Crimes in LAPD Data (Los Angeles Times)

- [Times Investigation: LAPD misclassified nearly 1,200 violent crimes as minor offenses](https://www.latimes.com/local/la-me-crimestats-lapd-20140810-story.html)
- [LAPD underreported serious assaults, skewing crime stats for 8 years](https://www.latimes.com/local/cityhall/la-me-crime-stats-20151015-story.html)
- [How we reported this story](https://www.latimes.com/local/cityhall/la-me-crime-stats-side-20151015-story.html)

## Definitions

>**Aggravated Assault:** An unlawful attack by one person upon another for the purpose of inflicting severe or aggravated bodily injury. This type of assault usually is accompanied by the use of a weapon or by means likely to produce death or great bodily harm.


>**Other Assault:** Simple, Not Aggravated. Includes all assaults which do not involve the use of a firearm, knife, cutting instrument, or other dangerous weapon and in which the victim did not sustain serious or aggravated injuries. 

## Our Data Sample

The dataset has hundreds of thousands of rows, but we will sample 100 from them for now: https://docs.google.com/spreadsheets/d/1LZ72b3cgVi7mhryMiromE3eT86DSnfna1cXjX-jLvGk/edit#gid=0

## Setup

Run the cell below to install the dependencies for this notebook.

In [1]:
# %pip install -r requirements.txt

## Load the data

In [2]:
%matplotlib inline
import csv, requests, os
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('data/df_sample_nolabels.csv')
print(f"Loaded {len(df)} rows")
df.head()

Loaded 375 rows


,Title,Poet,Year,URL,Poem Text
0,things that shine in the night,Rigoberto González,2016.0,https://poets.org/poem/things-shine-night,Fulgencio’s silver crown—when he snores\nthe m...
1,In a Landscape: IV,John Gallaher,2013.0,https://poets.org/poem/landscape-iv,"Now the scene changes, we say, and the next fe..."
2,Bird,Niki Herd,2020.0,https://poets.org/poem/bird,"Yesterday, at Shepherd and Gray, the parking l..."
3,"The Aeneid, Book VI [First, the sky and the ea...",Virgil,2006.0,https://poets.org/poem/aeneid-book-vi-first-sk...,"“First, the sky and the earth and the flowing ..."
4,"Pandemic: While home is an outbreak, we pass a...",Arianne True,2020.0,https://poets.org/poem/pandemic-while-home-out...,--\n--\n--\n--\n--\n--\nThis country has a way...


# You are the classifier 👈


Based on the definitions provided, categorize the data you have been assigned as `Other Assault` or `Aggrevated Assault`.

In [4]:
df.columns.tolist()

['Title', 'Poet', 'Year', 'URL', 'Poem Text']

## LLM as the classifier 🤖

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
openrouter_key = os.getenv("OPENROUTER_API_KEY")

In [6]:
import diskcache
cache = diskcache.Cache('./cache')  # stores in ./cache folder

This is a **pydantic model**. It defines what format I want the output to come back in. It's for an LLM feature called "Structured Outputs", but also works with other LLM tools. This gets the LLM to always return structured json data, which gets read into python as a class.

In [7]:
from pydantic import BaseModel

# This is a pydantic model. It defines what format I want the output to come back in
# It's for an OpenAI feature called "Structured Output", but also works with other LLM tools
class Classification(BaseModel):
    llm: bool
    reason: str

In [8]:
SYSTEM_PROMPT = """
Classify each poem individually into categories: either "true" if it is about immigration or "false" if it is not about immigration.

These are poems of all kinds of styles and from all manner of authors (not only American authors) so it is not the case that all poems that include non-English words or non-American place names are about immigration (though it may be that some poems about immigration also do include non-English words or non_US place names).

Also note that a poem about moving or change is also not necessarily about immigration directly; you should be able to articulate a specific reason you think the poem is about immigration.

Respond in JSON with two fields: "llm" (boolean) and "reason" (string).
"""

# Set the column from your CSV that contains the text to classify
TEXT_COLUMN = "Poem Text"

MODEL = "google/gemini-2.5-flash-lite"

In [9]:
from openai import OpenAI
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

@cache.memoize() # This is the diskcache! Now I will never hit the API twice with the same request!
def ask_model_to_classify(text_description, model):
  response = client.beta.chat.completions.parse(
    model=model,
    messages=[
      {
        "role": "system",
        "content": SYSTEM_PROMPT
      },
      {
        "role": "user",
        "content": text_description
      },
    ],
    response_format=Classification,
    temperature=0
  )

  return response.choices[0].message.content

In [10]:
import json
from tqdm.notebook import tqdm
tqdm.pandas()

df[MODEL] = df[TEXT_COLUMN].progress_apply(
    lambda x: ask_model_to_classify(x, model=MODEL) if pd.notna(x) else None
)
df['llm'] = df[MODEL].apply(lambda x: json.loads(x)['llm'] if x is not None else None)
df['reason'] = df[MODEL].apply(lambda x: json.loads(x)['reason'] if x is not None else None)

# delete raw model column
del df[MODEL]

df.head()

  0%|          | 0/375 [00:00<?, ?it/s]

,Title,Poet,Year,URL,Poem Text,llm,reason
0,things that shine in the night,Rigoberto González,2016.0,https://poets.org/poem/things-shine-night,Fulgencio’s silver crown—when he snores\nthe m...,True,The poem explicitly mentions 'México' and 'mig...
1,In a Landscape: IV,John Gallaher,2013.0,https://poets.org/poem/landscape-iv,"Now the scene changes, we say, and the next fe...",False,The poem describes personal experiences with i...
2,Bird,Niki Herd,2020.0,https://poets.org/poem/bird,"Yesterday, at Shepherd and Gray, the parking l...",False,The poem describes a personal experience of wi...
3,"The Aeneid, Book VI [First, the sky and the ea...",Virgil,2006.0,https://poets.org/poem/aeneid-book-vi-first-sk...,"“First, the sky and the earth and the flowing ...",False,The poem describes the creation of the univers...
4,"Pandemic: While home is an outbreak, we pass a...",Arianne True,2020.0,https://poets.org/poem/pandemic-while-home-out...,--\n--\n--\n--\n--\n--\nThis country has a way...,True,"The poem explicitly states, 'We left the dead ..."


## Inspect results

In [11]:
# Distribution of classifications
df['llm'].value_counts()

llm
False    349
True      22
Name: count, dtype: int64

In [12]:
# Browse results — show the text column and classifications
pd.set_option('display.max_colwidth', None)
df[[TEXT_COLUMN, 'llm', 'reason']].head(3)

,Poem Text,llm,reason
0,"Fulgencio’s silver crown—when he snores\nthe moon, coin of Judas, glaring\nat the smaller metals we call stars\nmy buckle\nthe tips of my boots\nthe stones in my kidneys\nan earring\na tear on the cheek\nthe forked paths of a zipper\nthe blade of the pocketknife triggering open\nthe blade of the pocketknife seducing the orange\nthe blade of the pocketknife salivating\nthe blade of the pocketknife\nthe word México\nthe word migra",True,"The poem explicitly mentions 'México' and 'migra,' which is a common term for immigration authorities in the United States. The imagery of 'forked paths' and the focus on metallic objects (crown, moon, buckle, boots, stones, earring, zipper, pocketknife) could symbolize the difficult and often harsh journey or experiences associated with migration."
1,"Now the scene changes, we say, and the next few years are quiet. It’s another curse, the inverse of the “interesting times” the Chinese were said to go on so about. Nevertheless, there it is, as the emptiness needs a something in order to be defined as empty, which means we spend the next few years talking about other years, as if that’s what’s important. Maybe that is what’s important. It was terrible, the hospital stay. The children. Not the children in the abstract way, but those times worried that this would go wrong, or that, and then things do go wrong and it almost feels like we’d wished for it to happen, so not only do we have to go through this terrible time, but we also have to keep reminding ourselves that we didn’t wish for it. It’s Problem One. And there’s our two-year-old son strapped to a board with an IV, crying. And doesn’t it feel like a formal device then? As if expecting it were the same—or is the same—as willing it, but then almost willing it anyway, saying something like, “Please God, or whomever, get it over with already . . .” if the world isn’t going to be a museum only, as museums keep calling out that there’s so much more to find in the past, like ourselves, for instance. The simplification of our forms. The question of why it might be important to save our dinnerware, or Yo-yos. We have these accidents in common: last night I was pulling a filing cabinet upstairs on a hand truck, and at the 90 degree turn it fell on top of me and I had to hold it like that, one wheel on the stair, one in mid-air. So I had some time on my hands, waiting for Robin to get home. They say that if you relax, lying there is 80% as restful as sleep. And knowing how to relax is key, they say. Here’s a guess: we will sit on a wooden lawn-chair in the sun, and we will like it. We will run the numbers and think it sounds like a good proposition. We will consult a map, even ask directions. The sun’s out right now, in fact, and it’s all a matter of doing the next big thing. Driving home, say. And then it’s a manner of having done something. Driving past the car wash. Yes, forcing a matter of doing the next thing, which is filling out the accident report, while the old man who hit my pickup is crying in the street. And then I’m walking around, picking up the fender and light pieces and putting them in the bed.",False,"The poem describes personal experiences with illness, accidents, and the mundane aspects of life. There are no references to migration, displacement, or the challenges associated with moving to a new country."
2,"Yesterday, at Shepherd and Gray, the parking lot was\nfilled with birds, black birds, actually grackles. It was a grackle\nlot; instead of a bumper on a car, there were ten grackles, instead\nof a sunroof, fifty grackles sat high, their bodies shimmers\nunder cheap strip mall lights as shoppers delayed their spending\nto pull out phones and take shots, such spectators we were,\nlike that summer in July, when I was left again\nto wonder who was the child and who the adult,\nthat Sunday evening that hung in the air like bug spray\nwhen my father, the one who fed me and gave me his last name,\nstood two sto

In [13]:
df.to_csv('data/df_llm_labels_1.csv', index=False)